# Designer Agent Notebook

This notebook showcases the Designer Agent, the primary interface for code generation.

## Purpose
The Designer Agent is responsible for translating user requirements into quantum circuits. This notebook demonstrates:

1.  **Agent Setup**: Initializing the Designer Agent with RAG capabilities.
2.  **Task Execution**: Sending natural language tasks (e.g., "Create a Teleportation circuit") to the agent.
3.  **Code Production**: Verifying that the agent produces syntactically correct Cirq code based on the input.

## Usage
Use this notebook to interact with the Designer Agent and generate initial circuit implementations.


In [1]:
import sys
import os
from pathlib import Path

# Add project root to path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.cirq_rag_code_assistant.config import get_config, setup_logging
from src.rag.knowledge_base import KnowledgeBase
from src.rag.retriever import Retriever
from src.rag.generator import Generator
from src.agents.designer import DesignerAgent

# Setup logging
setup_logging()

2026-03-26 11:09:01 | INFO     | src.cirq_rag_code_assistant.config.logging:setup_all:127 | Logging configuration completed


### Initialize RAG Components
We need to initialize the underlying RAG components first:
1. Load the KnowledgeBase and its entries
2. Build or load the vector index
3. Create Retriever and Generator

In [2]:
# Define paths
KNOWLEDGE_BASE_DIR = project_root / "data" / "knowledge_base"
VECTOR_INDEX_PATH = project_root / "data" / "models" / "vector_index"

# Initialize Knowledge Base and load entries
kb = KnowledgeBase(knowledge_base_path=KNOWLEDGE_BASE_DIR)
num_loaded = kb.load_from_directory()
print(f"Loaded {num_loaded} entries from knowledge base")

# Try to load existing index, or build new one
try:
    kb.load_index(VECTOR_INDEX_PATH)
    print(f"Loaded vector index: {kb.vector_store.size()} entries")
except FileNotFoundError:
    print("No existing index found, building new one...")
    kb.index_entries()
    kb.save_index(VECTOR_INDEX_PATH)
    print(f"Built and saved vector index: {kb.vector_store.size()} entries")

# Initialize RAG components (uses Ollama by default)
retriever = Retriever(knowledge_base=kb, use_hybrid_scoring=True)

try:
    generator = Generator(retriever=retriever)
    print("\n✅ RAG components initialized.")
except Exception as e:
    print(f"\n⚠️ Error initializing Generator: {e}")
    print("Make sure Ollama is running locally.")
    generator = None

2026-03-26 11:09:01 | INFO     | config.config_loader:load:101 | ✅ Loaded configuration from D:\University\Uni\Huawei ICT Compeition\Cirq-RAG-Code-Assistant\config\config.json


2026-03-26 11:09:01,581 - botocore.credentials - INFO - credentials.py:1252 - Found credentials in environment variables.


2026-03-26 11:09:01 | INFO     | src.rag.embeddings:_init_aws:124 | Using AWS Bedrock Embeddings: amazon.nova-2-multimodal-embeddings-v1:0, dim=1024
2026-03-26 11:09:01 | INFO     | src.rag.vector_store:_init_faiss:139 | Initialized FAISS index
2026-03-26 11:09:01 | INFO     | src.rag.vector_store:__init__:120 | Initialized VectorStore with faiss backend
2026-03-26 11:09:01 | INFO     | src.rag.vector_store:__init__:121 | Embedding dimension: 1024
2026-03-26 11:09:01 | INFO     | src.rag.knowledge_base:__init__:100 | Initialized KnowledgeBase with 0 entries
2026-03-26 11:09:01 | INFO     | src.rag.knowledge_base:load_from_jsonl:116 | Loading knowledge base from D:\University\Uni\Huawei ICT Compeition\Cirq-RAG-Code-Assistant\data\knowledge_base\curated_designer_examples.jsonl
2026-03-26 11:09:01 | INFO     | src.rag.knowledge_base:load_from_jsonl:129 | Loaded 107 entries from D:\University\Uni\Huawei ICT Compeition\Cirq-RAG-Code-Assistant\data\knowledge_base\curated_designer_examples.js

Loaded 140 entries from knowledge base
Loaded vector index: 140 entries

✅ RAG components initialized.


### Initialize Designer Agent
The Designer Agent combines retrieval and generation to produce Cirq code.

In [3]:
if generator is not None:
    # Initialize Designer Agent
    designer = DesignerAgent(retriever=retriever, generator=generator)
    print("✅ Designer Agent initialized.")
else:
    designer = None
    print("⚠️ Cannot initialize Designer Agent without Generator.")
    print("Make sure Ollama is running locally.")

2026-03-26 11:09:01 | INFO     | src.agents.base_agent:__init__:78 | Initialized DesignerAgent agent


✅ Designer Agent initialized.


### Generate Circuit
Let's ask the agent to design a circuit.

In [4]:
if designer is not None:
    task = {
        "query": "Create a circuit for Quantum Teleportation",
        "algorithm": "teleportation"
    }

    try:
        result = designer.execute(task)
        
        if result['success']:
            print("✅ Successfully generated circuit!")
            print("\nCode:")
            print("-" * 40)
            print(result['code'])
            print("-" * 40)
            print(f"\nAlgorithm: {result.get('algorithm', 'N/A')}")
            print(f"Context used: {result.get('context_used', 'N/A')} examples")
        else:
            print(f"❌ Failed to generate circuit: {result.get('error')}")
            if 'code' in result:
                print("\nGenerated code (with errors):")
                print(result['code'])
            
    except Exception as e:
        print(f"Error executing task: {e}")
else:
    print("Designer Agent not available.")

2026-03-26 11:09:01 | INFO     | src.rag.generator:generate:210 | Retrieving context for query: Create a circuit for Quantum Teleportation...
2026-03-26 11:09:03 | INFO     | src.rag.generator:generate:251 | Generating code using aws/arn:aws:bedrock:us-east-1:626230557862:application-inference-profile/lupvf1p1uqzg
2026-03-26 11:09:15 | INFO     | src.rag.generator:generate:319 | ✅ Code generation completed


✅ Successfully generated circuit!

Code:
----------------------------------------
import cirq

# Qubits: message (Alice's qubit to teleport), alice (entangled), bob (entangled)
message, alice, bob = cirq.LineQubit.range(3)

circuit = cirq.Circuit()

# Step 1: Prepare the message qubit in some state (e.g., apply X and H for a non-trivial state)
circuit.append([
    cirq.X(message)**0.3,
    cirq.Y(message)**0.1
])

# Step 2: Create Bell pair between Alice and Bob
circuit.append([
    cirq.H(alice),
    cirq.CNOT(alice, bob)
])

# Step 3: Bell measurement on message and alice (Alice's side)
circuit.append([
    cirq.CNOT(message, alice),
    cirq.H(message)
])

# Step 4: Measure Alice's qubits
circuit.append([
    cirq.measure(message, key='m'),
    cirq.measure(alice, key='a')
])

# Step 5: Classical corrections on Bob's qubit based on measurement results
# Apply X to bob if alice measurement is 1
circuit.append(cirq.X(bob).on(bob).with_classical_controls('a'))
# Apply Z to bob if messa

### Try More Examples
Let's try a few more circuit generation tasks.

In [5]:
if designer is not None:
    test_tasks = [
        {"query": "Create a Bell state circuit", "algorithm": "bell_state"},
        {"query": "Implement Grover's search for 2 qubits", "algorithm": "grover"},
        {"query": "Build a 3-qubit GHZ state", "algorithm": "ghz_state"},
    ]
    
    for i, task in enumerate(test_tasks, 1):
        print(f"\n{'='*60}")
        print(f"Task {i}: {task['query']}")
        print('='*60)
        
        try:
            result = designer.execute(task)
            if result['success']:
                print("✅ Success")
                # Show first 10 lines of code
                code_lines = result['code'].split('\n')[:10]
                print("Code preview:")
                for line in code_lines:
                    print(f"  {line}")
                if len(result['code'].split('\n')) > 10:
                    print("  ...")
            else:
                print(f"❌ Failed: {result.get('error')}")
        except Exception as e:
            print(f"Error: {e}")

2026-03-26 11:09:15 | INFO     | src.rag.generator:generate:210 | Retrieving context for query: Create a Bell state circuit...



Task 1: Create a Bell state circuit


2026-03-26 11:09:16 | INFO     | src.rag.generator:generate:251 | Generating code using aws/arn:aws:bedrock:us-east-1:626230557862:application-inference-profile/lupvf1p1uqzg
2026-03-26 11:09:22 | INFO     | src.rag.generator:generate:319 | ✅ Code generation completed
2026-03-26 11:09:22 | INFO     | src.rag.generator:generate:210 | Retrieving context for query: Implement Grover's search for 2 qubits...


✅ Success
Code preview:
  import cirq
  
  # Create two qubits
  q0, q1 = cirq.LineQubit.range(2)
  
  # Build the Bell state circuit: |Φ+⟩ = (|00⟩ + |11⟩) / √2
  circuit = cirq.Circuit([
      cirq.H(q0),          # Apply Hadamard to put q0 in superposition
      cirq.CNOT(q0, q1),   # Entangle q0 and q1
      cirq.measure(q0, q1, key='result')  # Measure both qubits
  ...

Task 2: Implement Grover's search for 2 qubits


2026-03-26 11:09:23 | INFO     | src.rag.generator:generate:251 | Generating code using aws/arn:aws:bedrock:us-east-1:626230557862:application-inference-profile/lupvf1p1uqzg
2026-03-26 11:09:36 | INFO     | src.rag.generator:generate:319 | ✅ Code generation completed
2026-03-26 11:09:36 | INFO     | src.rag.generator:generate:210 | Retrieving context for query: Build a 3-qubit GHZ state...


✅ Success
Code preview:
  import cirq
  import numpy as np
  
  # 2 qubits for Grover's search (4 possible states: 00, 01, 10, 11)
  qubits = cirq.LineQubit.range(2)
  q0, q1 = qubits
  
  # Target state to find: |11> (marked state)
  # Oracle: flips phase of |11> using CZ gate
  def oracle(q0, q1):
  ...

Task 3: Build a 3-qubit GHZ state


2026-03-26 11:09:37 | INFO     | src.rag.generator:generate:251 | Generating code using aws/arn:aws:bedrock:us-east-1:626230557862:application-inference-profile/lupvf1p1uqzg
2026-03-26 11:09:45 | INFO     | src.rag.generator:generate:319 | ✅ Code generation completed


✅ Success
Code preview:
  import cirq
  
  # Define 3 qubits
  qubits = cirq.LineQubit.range(3)
  q0, q1, q2 = qubits
  
  # Build the GHZ circuit
  circuit = cirq.Circuit()
  
  # Apply Hadamard to the first qubit to create superposition
  ...
